# 🎙️ RVC Voice Training - 单田芳声音模型

## 使用说明

1. **运行此笔记本** - 点击上方 "播放" 按钮逐个运行单元格
2. **上传音频** - 在左侧文件面板上传单田芳评书音频（建议30分钟以上）
3. **开始训练** - 运行训练单元格
4. **下载模型** - 训练完成后下载模型文件

## 推荐音频来源
- 喜马拉雅 (ximalaya.com)
- 蜻蜓FM (qingting.fm)
- 或购买正版音频

In [ ]:
# @title 📦 安装依赖
import os
from google.colab import files

# 克隆 RVC 项目
if not os.path.exists('/content/Retrieval-based-Voice-Conversion'):
    print("🔄 克隆 RVC 仓库...")
    !git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion.git
else:
    print("✅ RVC 已存在")

# 进入目录
%cd /content/Retrieval-based-Voice-Conversion

# 安装依赖
print("📚 安装 Python 依赖...")
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q praat-parselmouth

print("✅ 安装完成!")

In [ ]:
# @title 📁 上传音频文件
# @markdown > 上传您的单田芳评书音频文件（支持 MP3, WAV, FLAC, M4A）
# @markdown > 建议：30分钟以上音频，越多越好

uploaded = files.upload()

# 创建音频目录
!mkdir -p /content/audio_source

# 移动上传的文件
for filename in uploaded.keys():
    print(f"📄 已上传: {filename}")
    !mv "$filename" /content/audio_source/

In [ ]:
# @title 🔧 预处理音频
# @markdown 将音频转换为训练格式 (16kHz, 单声道)

import subprocess
import os

!mkdir -p /content/audio_processed

audio_dir = "/content/audio_source"
output_dir = "/content/audio_processed"

# 获取所有音频文件
audio_files = []
for ext in ['.mp3', '.wav', '.flac', '.m4a', '.ogg']:
    audio_files.extend([f for f in os.listdir(audio_dir) if f.endswith(ext)])

print(f"📊 找到 {len(audio_files)} 个音频文件")

# 转换每个文件
for i, audio_file in enumerate(audio_files):
    input_path = f"{audio_dir}/{audio_file}"
    output_path = f"{output_dir}/{os.path.splitext(audio_file)[0]}.wav"
    
    print(f"\n[{i+1}/{len(audio_files)}] 处理: {audio_file}")
    
    # 转换为 16kHz 单声道 WAV
    cmd = f'ffmpeg -y -i "{input_path}" -ar 16000 -ac 1 -acodec pcm_s16le "{output_path}"'
    result = subprocess.run(cmd, shell=True, capture_output=True)
    
    if result.returncode == 0:
        print(f"  ✅ 已转换")
    else:
        print(f"  ❌ 失败: {result.stderr.decode()[:100]}")

print(f"\n✅ 音频预处理完成! 输出目录: {output_dir}")

In [ ]:
# @title 🚀 训练模型
# @markdown 配置训练参数并开始训练

model_name = "tianfangfang" # @param {type:"string"}
epochs = 100 # @param {type:"integer", "min":10, "max":500}
batch_size = 32 # @param {type:"integer", "min":1, "max":64}
pitch_method = "harvest" # @param ["harvest", "dio", "crepe"]

print("=" * 60)
print(f"🎯 训练配置:")
print(f"   模型名称: {model_name}")
print(f"   训练轮数: {epochs}")
print(f"   批次大小: {batch_size}")
print(f"   音高方法: {pitch_method}")
print("=" * 60)

# 创建输出目录
!mkdir -p /content/output/$model_name

# 运行训练
# 注意: RVC 训练命令可能因版本而异
print("\n⚠️  注意: 以下是 RVC v1 的训练命令。如果失败，请查看 RVC GitHub 获取最新命令。")

train_cmd = f"""
python train.py \
    --name {model_name} \
    --epochs {epochs} \
    --batch_size {batch_size} \
    --f0method {pitch_method} \
    --train_path /content/audio_processed \
    --output_path /content/output/{model_name}
"""

print(f"训练命令: {train_cmd}")
# !python train.py --help  # 查看可用参数

In [ ]:
# @title 📥 下载训练好的模型
# @markdown 训练完成后，运行此单元格下载模型文件

import os
import shutil
from google.colab import files

model_name = "tianfangfang" # @param {type:"string"}

model_dir = f"/content/output/{model_name}"

if os.path.exists(model_dir):
    # 创建压缩包
    zip_name = f"{model_name}.zip"
    print(f"📦 压缩模型文件: {zip_name}")
    shutil.make_archive(model_name, 'zip', model_dir)
    
    # 下载
    print("⬇️  开始下载...")
    files.download(f"/content/{zip_name}")
else:
    print(f"❌ 模型目录不存在: {model_dir}")
    print("请先运行训练!")

---

## 💡 提示

1. **训练时间**: 约 30分钟-2小时（取决于GPU和音频量）
2. **Colab 会话**: 如果训练时间过长，Colab 可能会断开。建议使用 **Colab Pro** 或 **RunPod**
3. **替代方案**: 如果 RVC 训练太复杂，可以直接使用 **Azure TTS** 生成配音
